# 03 · Final model, Kaggle submission, deployment and monitoring
Trains on all the data, scores the official test set on Kaggle (late submission), exports the model for the **FastAPI** service,
calls the running API, and runs a **drift check**. Writes `reports/results_summary.md`. **Runtime: CPU**, about 10 min.

In [ ]:
# ---------- Setup: run this cell first (CPU runtime is enough) ----------
import os, sys, subprocess, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FC_SMOKE") == "1"
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if SMOKE:
    PROJECT_DIR = os.environ["FC_PROJECT_DIR"]
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = sorted(glob.glob("/content/drive/MyDrive/**/forecast/src/forecast/models.py", recursive=True), key=len)
    assert hits, "Upload the 'forecast' folder to your Google Drive first."
    PROJECT_DIR = str(Path(hits[0]).parents[2])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate", "fastapi", "uvicorn", "httpx"], check=True)
else:
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, f"{PROJECT_DIR}/src"); sys.path.insert(0, PROJECT_DIR)
ART, FIG = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures"
DATA = "/content/data/rossmann" if ON_COLAB else f"{PROJECT_DIR}/data"
for d in (ART, FIG, DATA): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from forecast import data as D, models as M
from forecast.features import FEATURES, build_features, fit_store_stats
if SMOKE:
    train, test, store = D.make_synthetic(25)
else:
    train, test, store = D.load(D.download(DATA))
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=str)
def load_json(name):
    return json.load(open(f"{ART}/{name}.json"))
print(f"train {train.shape} {train.Date.min().date()}..{train.Date.max().date()} | test {test.shape} | stores {store.shape[0]}")

In [ ]:
bt = load_json("backtest")
t0 = time.time()
final = M.fit_lgbm(train, store, num_boost_round=bt["n_rounds"])
test_pred = final.predict(test, store) * bt["factor"]
sub = pd.DataFrame({"Id": test.Id, "Sales": test_pred}).sort_values("Id")
sub.to_csv(f"{ART}/submission.csv", index=False)
print(f"trained on {len(train):,} rows in {time.time()-t0:.0f}s; submission {sub.shape}"); sub.head()

## 1. Official score: Kaggle late submission (private leaderboard = 52% of the test set)

In [ ]:
KAGGLE = {}
if not SMOKE:
    try:
        r = subprocess.run(["kaggle", "competitions", "submit", "-c", D.COMPETITION, "-f", f"{ART}/submission.csv",
                            "-m", f"LightGBM global model, {bt['n_rounds']} rounds, factor {bt['factor']:.3f}"],
                           capture_output=True, text=True); print(r.stdout[-300:], r.stderr[-300:])
        for _ in range(30):                      # wait for scoring
            time.sleep(10)
            out = subprocess.run(["kaggle", "competitions", "submissions", "-c", D.COMPETITION, "--csv"],
                                 capture_output=True, text=True).stdout
            import io
            subs = pd.read_csv(io.StringIO(out)); subs.columns = [c.strip() for c in subs.columns]
            latest = subs.iloc[0]
            if str(latest.get("status", "")).lower().endswith("complete") and pd.notna(latest.get("privateScore")):
                KAGGLE = {"public_rmspe": float(latest["publicScore"]), "private_rmspe": float(latest["privateScore"])}
                break
        print("Kaggle scores:", KAGGLE or "not available yet - check the competition's 'Submissions' page")
    except Exception as e:  # noqa: BLE001
        print("Submission failed:", e, "-> upload artifacts/submission.csv manually via 'Late Submission' on Kaggle")

## 2. Export the model and serve it with FastAPI

In [ ]:
Q = {float(k): v for k, v in bt["conformal_q"].items()}
MODEL_DIR = f"{ART}/model"
M.save_model(final, store, Q, {"version": "1.0", "trained_until": str(train.Date.max().date()), "backtest": bt["summary"],
                                "conformal_coverage": bt["coverage"], "kaggle_score": KAGGLE, "factor": bt["factor"]}, MODEL_DIR)
os.environ["MODEL_DIR"] = MODEL_DIR
import importlib, app.main as api; importlib.reload(api)
from fastapi.testclient import TestClient          # in-process client: same code path as the deployed service
client = TestClient(api.app)
print(client.get("/health").json())
s1 = int(test.Store.iloc[0])
rows = [{"store": s1, "date": str(d.date()), "promo": int(p), "open": int(o)} for d, p, o in
        test[test.Store == s1].sort_values("Date")[["Date", "Promo", "Open"]].head(7).itertuples(index=False)]
pd.DataFrame(client.post("/predict", json={"rows": rows}).json()["predictions"])

**Deploying it:** `docker build -t rossmann-forecast . && docker run -p 8000:8000 rossmann-forecast`, then open
`http://localhost:8000/docs` for the interactive Swagger UI. GitHub Actions runs the unit and API tests on every push.

## 3. Monitoring: has the input data drifted? (population stability index, training vs. test period)

In [ ]:
ref = build_features(train[train.Date > train.Date.max() - pd.Timedelta(days=365)], store, final.stats)
cur = build_features(test, store, final.stats)
num = [f for f in final.features if f not in ("StoreType", "Assortment", "StateHoliday", "Store")]
drift = pd.Series({f: M.psi(ref[f], cur[f]) for f in num}).sort_values(ascending=False)
drift.head(10).plot.barh(figsize=(6, 4), color=["#C44E52" if v > 0.25 else "#DD8452" if v > 0.1 else "#55A868" for v in drift.head(10)])
plt.axvline(0.1, ls="--", c="k", lw=1); plt.axvline(0.25, ls="--", c="r", lw=1); plt.gca().invert_yaxis()
plt.title("Feature drift (PSI): last year vs. forecast period"); savefig("03_drift")
drift.head(10).round(3).to_frame("PSI")

## 4. Auto-generated results summary

In [ ]:
eda = load_json("eda"); abl = pd.read_csv(f"{ART}/ablation.csv"); folds = pd.read_csv(f"{ART}/backtest_folds.csv")
summ = folds.groupby("model")[["rmspe", "wape"]].agg(["mean", "std"]).round(4)
lg = summ.loc["LightGBM (global)"]; nb_ = summ.loc["Seasonal naive (same weekday, last 6 wks)"]
cov = pd.DataFrame(bt["coverage"])
lines = ["# Results summary (auto-generated by notebook 03)", "",
 f"* Data: {eda['n_rows']:,} store-days, {eda['n_stores']} stores; promo lift +{eda['promo_lift']:.1%}; {eda['store_spread']:.0f}x spread in store size",
 f"* **Backtest (3 rolling 48-day folds): LightGBM RMSPE {lg[('rmspe','mean')]:.4f} ± {lg[('rmspe','std')]:.4f} vs seasonal naive "
 f"{nb_[('rmspe','mean')]:.4f}** (WAPE {lg[('wape','mean')]:.4f} vs {nb_[('wape','mean')]:.4f})",
 f"* Bias-correction factor {bt['factor']:.3f} (fit on earlier folds)",
 "* Conformal coverage on unseen fold: " + ", ".join(f"{int(r['nominal']*100)}% nominal -> {r['coverage_overall']:.1%}" for r in bt["coverage"]),
 f"* **Kaggle (late submission): {KAGGLE}**" if KAGGLE else "* Kaggle score: (not available, see notebook)",
 f"* Drift: {int((drift > 0.25).sum())} features with PSI > 0.25, {int(((drift > 0.1) & (drift <= 0.25)).sum())} with 0.1-0.25",
 "", "## Backtest", summ.to_markdown(), "", "## Per fold", folds.round(4).to_markdown(index=False),
 "", "## Feature-group ablation (latest fold)", abl.round(4).to_markdown(index=False),
 "", "## Conformal intervals", cov.round(3).to_markdown(index=False),
 "", "## Top drift (PSI)", drift.head(10).round(3).to_frame("PSI").to_markdown()]
open(f"{PROJECT_DIR}/reports/results_summary.md", "w").write("\n".join(lines)); print("\n".join(lines[:9]))